In [ ]:
import os

data_dir = os.path.abspath("data/public")
project_dir = os.path.abspath("data/project")
chrombpnet_dir = os.path.join(project_dir, "chrombpnet")

In [ ]:
import torch
from bpnetlite.chrombpnet import ChromBPNet
from bpnetlite.bpnet import BPNet
from tangermeme.ersatz import shuffle
from tangermeme.predict import predict
import tangermeme
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import h5py
import torch
import torch.nn as nn
import pyranges
import pysam
from liftover import get_lifter
import copy
import itertools
import json
from tqdm.notebook import tqdm
from tangermeme.io import extract_loci
from tangermeme.deep_lift_shap import deep_lift_shap
from memelite.fimo import fimo, read_meme
import pickle
import scanpy as sc
import anndata as ad
import pysam
from adpbulk import ADPBulk
import edgepython as ep
import pyBigWig
import random
from tangermeme.utils import one_hot_encode
import torch.optim as optim
from collections import defaultdict
import torch.nn.functional as F

## Finetune for cell type differences

## Define peak set to use for finetuning

In [ ]:
# Setup
data_path = f"{chrombpnet_dir}/IC_Final_Raw_AnnData.h5ad"
celltypes = ["acinar", "activated_stellate", "alpha", "beta", "cycling", "delta","ductal", "endothelial", "epsilon", "lymphoid", "mast", "myeloid", "pp", "schwann", "quiescent_stellate"]
celltypes_label = 'labels'
donor_labels = 'donor_id'

In [ ]:
# Import the data object
adata = sc.read_h5ad(data_path)

In [ ]:
# Pseudobulks per celltype
for celltype in celltypes:
    # Subset the anndata object
    adata_ct = adata[adata.obs['labels'] == celltype,:].copy()
    
    # Setup for pseudobulk
    adpb = ADPBulk(adata_ct, donor_labels)
    
    # Perform the pseudobulking
    pseudobulk_matrix = adpb.fit_transform().T
    pseudobulk_matrix.columns = celltype + "_" + pseudobulk_matrix.columns
    
    # Retrieve the sample meta data (useful for easy incorporation with edgeR)
    sample_meta = adpb.get_meta()
    sample_meta['SampleName'] = celltype + "_" + sample_meta['SampleName']
    sample_meta['Celltype'] = celltype
    
    # Concat
    if celltype == celltypes[0]:
        metadata = sample_meta.copy()
        counts = pseudobulk_matrix.copy()
    else:
        metadata = pd.concat([metadata, sample_meta])
        counts = pd.concat([counts,pseudobulk_matrix], axis=1)

In [ ]:
# Estimate size factors using the full dataset
y = ep.make_dgelist(counts=counts, group=metadata['Celltype'])
y = ep.calc_norm_factors(y)
samples = y['samples']
samples = samples.set_index(metadata['SampleName']).copy()

In [ ]:
# Create dict for storing results
peakDict = {}
for celltypeA in celltypes:
    otherCelltypes = list(set(celltypes) - set([celltypeA]))
    for celltypeB in otherCelltypes:
        pair = celltypeA + "_" + celltypeB
        peakDict[pair] = {}

In [ ]:
# Find differential accessibile peaks per cell type
for celltypeidx in tqdm(range(len(celltypes))):
    # Subset the anndata object
    celltype = celltypes[celltypeidx]
    adata_ct = adata[adata.obs['labels'] == celltype,:].copy()

    # Subset to peaks open in 10% of that cell type
    counts_subset = counts.loc[np.array(counts.index.tolist())[np.argwhere((np.array((adata_ct.X > 0).sum(axis = 0))[0,:] / adata_ct.shape[0]) >= 0.1)[:,0]]].copy()

    # Define other cell types
    otherCelltypes = list(set(celltypes) - set([celltype]))

    # Loop over other celltypes
    for otherCelltype in otherCelltypes:
        # Define name of pair
        pair = celltype + "_" + otherCelltype
    
        # Structure the metadata
        metadata_ct = metadata.copy()
        metadata_ct['Test'] = 'Control'
        metadata_ct.iloc[np.argwhere(metadata_ct['Celltype'] == celltype)[:,0],3] = "Target"
        metadata_ct = metadata_ct[metadata_ct['Celltype'].isin([celltype, otherCelltype])].copy()
        
        # Extract the pair of counts
        counts_pair = counts_subset.T.loc[metadata_ct['SampleName']].T.copy()
        
        # Make a DGEList object
        y = ep.make_dgelist(counts=counts_pair, group=metadata_ct['Test'])
        
        # Set the design matrix
        design = ep.utils.model_matrix("~ Test", metadata_ct)
        
        # Inject sample information
        y['samples'] = samples.loc[metadata_ct['SampleName']].copy()
        
        # Estimate dispersions
        y = ep.estimate_disp(y, design = design)
        
        # Fit model
        fit = ep.glm_fit(y, design)
        
        # LRT testing
        res = ep.glm_lrt(fit)
        
        # Extract results
        top = ep.top_tags(res, n=len(counts_subset))
        top = top['table']
        
        # Define hits and controls
        hits = top[top['logFC'] >= np.log(1.3)].copy()
        hits = hits[hits['FDR'] <= 0.01].copy()
        controls = top[top['logFC'].abs() < np.log2(1.3)].copy()
        controls = controls[controls['FDR'] > 0.05].copy()
        
        # Store results
        peakDict[pair]["hits"] = np.array(counts_subset.index.tolist())[hits.index.tolist()].tolist()
        peakDict[pair]["controls"] = np.array(counts_subset.index.tolist())[controls.index.tolist()].tolist()

# Write to pickle file
with open(f"{chrombpnet_dir}/data/differential_accessibility.pkl", "wb") as f:
    pickle.dump(peakDict, f)

### Setup models

In [ ]:
# Modified from here: https://github.com/jmschrei/bpnet-lite/blob/master/bpnetlite/bpnet.py#L551
def import_chrombpnet_accessibility(filename, model_name = 'wo_bias'):
    with h5py.File(filename, "r") as h5:
        w = h5['model_weights']
        if model_name == 'wo_bias':
            prefix = 'wo_bias_'
            model_name = 'model_wo_bias'
        else:
            prefix = ''
            model_name = 'model'
        namer = lambda model_name, prefix, suffix: '{0}/{1}{2}'.format(model_name, prefix, suffix)
        k, b = 'kernel:0', 'bias:0'
        n_layers = 0
        for layer_name in w[model_name].keys():
            try:
                idx = int(layer_name.split("_")[-1].replace("conv", ""))
                n_layers = max(n_layers, idx)
            except:
                pass
        name = namer(model_name, prefix, "bpnet_1conv")
        n_filters = w[name][k].shape[2]

        model = BPNet(n_layers=n_layers, n_filters=n_filters, n_outputs=1, n_control_tracks=0)
        convert_w = lambda x: torch.nn.Parameter(torch.tensor(x[:]).permute(2, 1, 0))
        convert_b = lambda x: torch.nn.Parameter(torch.tensor(x[:]))

        iname = namer(model_name, prefix, 'bpnet_1st_conv')

        model.iconv.weight = convert_w(w[iname][k])
        model.iconv.bias = convert_b(w[iname][b])
        model.iconv.padding = ((21 - 1) // 2,)

        for i in range(1, n_layers+1):
            lname = namer(model_name, prefix, 'bpnet_{}conv'.format(i))
            model.rconvs[i-1].weight = convert_w(w[lname][k])
            model.rconvs[i-1].bias = convert_b(w[lname][b])

        if model_name == 'model_wo_bias':
            prefix = 'wo_bias_bpnet_'
        fname = namer(model_name, prefix, 'prof_out_precrop')
        model.fconv.weight = convert_w(w[fname][k])
        model.fconv.bias = convert_b(w[fname][b])
        model.fconv.padding = ((75 - 1) // 2,)
        name = namer(model_name, prefix, "logcount_predictions")
        model.linear.weight = torch.nn.Parameter(torch.tensor(w[name][k][:].T))
        model.linear.bias = convert_b(w[name][b])
        return model

In [ ]:
# Modified from here: https://github.com/jmschrei/bpnet-lite/blob/master/bpnetlite/bpnet.py#L551
import re
def import_chrombpnet_bias(filename):
    with h5py.File(filename, "r") as h5:
        w = h5['model_weights']
        namer = lambda prefix, suffix: '{0}{1}/{0}{1}'.format(prefix, suffix)
        prefix = ""
        k, b = 'kernel:0', 'bias:0'
        nums = []
        for layer_name in w.keys():
            try:
                match = re.match(r"bpnet_(\d+)conv", layer_name)
                if match:
                    nums.append(int(match.group(1)))
            except:
                pass
        n_layers = max(nums)
        name = namer(prefix, "bpnet_1conv")
        n_filters = w[name][k].shape[2]

        model = BPNet(n_layers=n_layers, n_filters=n_filters, n_outputs=1, n_control_tracks=0)
        convert_w = lambda x: torch.nn.Parameter(torch.tensor(x[:]).permute(2, 1, 0))
        convert_b = lambda x: torch.nn.Parameter(torch.tensor(x[:]))

        iname = namer(prefix, 'bpnet_1st_conv')

        model.iconv.weight = convert_w(w[iname][k])
        model.iconv.bias = convert_b(w[iname][b])
        model.iconv.padding = ((21 - 1) // 2,)

        for i in range(1, n_layers+1):
            lname = namer(prefix, 'bpnet_{}conv'.format(i))
            model.rconvs[i-1].weight = convert_w(w[lname][k])
            model.rconvs[i-1].bias = convert_b(w[lname][b])

        fname = namer(prefix, 'prof_out_precrop')
        model.fconv.weight = convert_w(w[fname][k])
        model.fconv.bias = convert_b(w[fname][b])
        model.fconv.padding = ((75 - 1) // 2,)
        name = namer(prefix, "logcount_predictions")
        model.linear.weight = torch.nn.Parameter(torch.tensor(w[name][k][:].T))
        model.linear.bias = convert_b(w[name][b])
        return model

In [ ]:
def freeze_layers(model, freeze_output_heads = True, freeze_initial = True, freeze_layers = []):
    # Freeze the bias model
    for param in model.bias.parameters():
        param.requires_grad = False

    # Freeze layers in accessibility model
    for name, param in model.named_parameters():
        # Count head
        if "linear" in name:
            if freeze_output_heads:
                param.requires_grad = False
            else:
                param.requires_grad = True
        
        # Profile head
        if "fconv" in name:
            if freeze_output_heads:
                param.requires_grad = False
            else:
                param.requires_grad = True
        
        # Initial convolution
        if "iconv" in name:
            if freeze_initial:
                param.requires_grad = False
            else:
                param.requires_grad = True

        # Convulational stack
        for i in range(7):
            if f"accessibility.rconvs.{i}." in name:
                if i in freeze_layers:
                    param.requires_grad = False
                else:
                    param.requires_grad = True
                
def custom_forward(model, seq):
    # Pass through bias model
    start, end = model.bias.trimming, seq.shape[2] - model.bias.trimming
    X_ = model.bias.irelu(model.bias.iconv(seq))
    for i in range(model.bias.n_layers):
        X_conv = model.bias.rrelus[i](model.bias.rconvs[i](X_))
        X_ = torch.add(X_, X_conv)
    bias_profile = model.bias.fconv(X_)[:, :, start:end]
    bias_count = model.bias.linear(torch.mean(X_[:, :, start-37:end+37], dim=2)).reshape(X_.shape[0], 1)

    # Pass through accessibility model
    start, end = model.accessibility.trimming, seq.shape[2] - model.accessibility.trimming
    X_ = model.accessibility.irelu(model.accessibility.iconv(seq))
    for i in range(model.accessibility.n_layers):
        X_conv = model.accessibility.rrelus[i](model.accessibility.rconvs[i](X_))
        X_ = torch.add(X_, X_conv)
    embedding = torch.mean(X_[:, :, start-37:end+37], dim=2)
    accessibility_profile = model.accessibility.fconv(X_)[:, :, start:end]
    accessibility_count = model.accessibility.linear(embedding).reshape(X_.shape[0], 1)
    
    # Adjusted count
    n0, n1 = accessibility_profile.shape[-1], bias_profile.shape[-1]
    w = (n1 - n0) // 2
    
    y_profile = torch.nn.functional.log_softmax(accessibility_profile + bias_profile[:, :, w:-w], dim=-1)
    y_counts = torch.logsumexp(torch.stack([accessibility_count, bias_count]), dim=0)

    # Returns
    return y_profile, y_counts, embedding

class ProjectionHead(nn.Module):
    def __init__(self, in_dim: int = 512, mid_dim: int = 128, out_dim: int = 32):
        super().__init__()
        self.W = nn.Sequential(
            nn.Linear(in_dim, mid_dim, bias = False),
            nn.ReLU(),
            nn.Linear(mid_dim, out_dim, bias = False)
        )
    def forward(self, x):
        return F.normalize(self.W(x), dim=1)

In [ ]:
def getBatch(
    batch_size: int = 32,
    celltypes: list = [],
    data: pd.DataFrame = pd.DataFrame(),
    sets: dict = {}, 
    inputlen: int = 2114,
    outputlen: int = 1000,
    max_jitter: int = 128,
    reverseComplement: bool = True,
    minPeaks: int = 20,
    shuffle: bool = True
    ):

    # Translation tables
    toN = str.maketrans({chr(i):"N" for i in range(256) if chr(i).upper() not in "ACGT"})
    comp = str.maketrans("ACGT", "TGCA")
    
    # Init groupMin and select cell type pairs untill hit a pair with more peaks than minPeaks
    groupMin = 0
    while groupMin < minPeaks:
        # Sample a random pair of celltypes
        celltypeA = random.choice(celltypes)
        celltypeB = random.choice(celltypes)
        while celltypeA == celltypeB:
            celltypeB = random.choice(celltypes)
    
        # Sample positive and negative peaks
        pair = celltypeA + "_" + celltypeB
        nHits = len(sets[pair]["hits"])
        nControls = len(sets[pair]["controls"])
        groupMin = min([nHits, nControls])

    # Get sampling size from each group
    if groupMin < batch_size // 2:
        batch_size_per_group = groupMin
    else:
        batch_size_per_group = batch_size // 2

    # Sample positives and negatives
    positives = random.sample(sets[pair]["hits"], k=batch_size_per_group)
    negatives = random.sample(sets[pair]["controls"], k=batch_size_per_group)

    # Sample jitters
    positive_jitter = random.sample(list(range(-max_jitter, max_jitter)), k=batch_size_per_group)
    negative_jitter = random.sample(list(range(-max_jitter, max_jitter)), k=batch_size_per_group)

    # Get sequences
    positive_sequences = [s[a:b].upper().translate(toN) for s, a, b in zip(peaks_df.loc[positives]['sequences'].values, max_jitter + np.array(positive_jitter), max_jitter + inputlen + np.array(positive_jitter))]
    negative_sequences = [s[a:b].upper().translate(toN) for s, a, b in zip(peaks_df.loc[negatives]['sequences'].values, max_jitter + np.array(negative_jitter), max_jitter + inputlen + np.array(negative_jitter))]
    
    # Get counts
    offset = (inputlen - outputlen) // 2
    positive_counts_celltypeA = np.asarray([s[a:b] for s, a, b in zip(peaks_df.loc[positives][celltypeA].values, max_jitter + offset + np.array(positive_jitter), max_jitter + offset + outputlen + np.array(positive_jitter))])
    negative_counts_celltypeA = np.asarray([s[a:b] for s, a, b in zip(peaks_df.loc[negatives][celltypeA].values, max_jitter + offset + np.array(negative_jitter), max_jitter + offset + outputlen + np.array(negative_jitter))])
    positive_counts_celltypeB = np.asarray([s[a:b] for s, a, b in zip(peaks_df.loc[positives][celltypeB].values, max_jitter + offset + np.array(positive_jitter), max_jitter + offset + outputlen + np.array(positive_jitter))])
    negative_counts_celltypeB = np.asarray([s[a:b] for s, a, b in zip(peaks_df.loc[negatives][celltypeB].values, max_jitter + offset + np.array(negative_jitter), max_jitter + offset + outputlen + np.array(negative_jitter))])

    # Reverse complement
    if reverseComplement:
        positive_rc = random.sample([True, False], counts=[batch_size_per_group, batch_size_per_group], k=batch_size_per_group)
        negative_rc = random.sample([True, False], counts=[batch_size_per_group, batch_size_per_group], k=batch_size_per_group)
        positive_sequences = [s.translate(comp)[::-1] if do_rc else s for s, do_rc in zip(positive_sequences, positive_rc)]
        negative_sequences = [s.translate(comp)[::-1] if do_rc else s for s, do_rc in zip(negative_sequences, negative_rc)]
        positive_counts_celltypeA[positive_rc] = positive_counts_celltypeA[positive_rc][:, ::-1]
        negative_counts_celltypeA[negative_rc] = negative_counts_celltypeA[negative_rc][:, ::-1]
        positive_counts_celltypeB[positive_rc] = positive_counts_celltypeB[positive_rc][:, ::-1]
        negative_counts_celltypeB[negative_rc] = negative_counts_celltypeB[negative_rc][:, ::-1]

    # Onehot encode sequences
    positive_sequences = torch.stack([one_hot_encode(s, ignore = ["R", "Y", "S", "W", "K", "M", "B", "D", "H", "V", "N"]) for s in positive_sequences])
    negative_sequences = torch.stack([one_hot_encode(s, ignore = ["R", "Y", "S", "W", "K", "M", "B", "D", "H", "V", "N"]) for s in negative_sequences])

    # Stack everything
    labels = torch.hstack([torch.zeros(batch_size_per_group), torch.ones(batch_size_per_group)])
    sequences = torch.vstack([positive_sequences, negative_sequences]).to(torch.float32)
    celltypeA_counts = torch.vstack([torch.tensor(positive_counts_celltypeA), torch.tensor(negative_counts_celltypeA)]).to(torch.float32)
    celltypeB_counts = torch.vstack([torch.tensor(positive_counts_celltypeB), torch.tensor(negative_counts_celltypeB)]).to(torch.float32)

    # Shuffle
    if shuffle:
        perm = torch.randperm(labels.size(0))
        labels = labels[perm]
        sequences = sequences[perm]
        celltypeA_counts = celltypeA_counts[perm]
        celltypeB_counts = celltypeB_counts[perm]
    
    return labels, sequences, celltypeA_counts, celltypeB_counts, celltypeA, celltypeB

In [ ]:
# Source: https://github.com/jmschrei/bpnet-lite/blob/master/bpnetlite/losses.py#L99
def MNLLLoss(logps, true_counts):
    log_fact_sum = torch.lgamma(torch.sum(true_counts, dim=-1) + 1)
    log_prod_fact = torch.sum(torch.lgamma(true_counts + 1), dim=-1)
    log_prod_exp = torch.sum(true_counts * logps, dim=-1)
    return -log_fact_sum + log_prod_fact - log_prod_exp

In [ ]:
def supervised_infonce(z1, z2, y, tau=0.1):
    B = z1.size(0)
    z = torch.cat([z1, z2], dim=0)        # (2B, D)
    sim = torch.matmul(z, z.T) / tau      # (2B, 2B)

    # mask self similarity
    logits_mask = ~torch.eye(2*B, dtype=torch.bool, device=z.device)
    sim = sim.masked_fill(~logits_mask, -1e9)

    # build supervised positives mask
    # anchor i (in z1) is positive with B+i (in z2) if y[i]==1
    pos_mask = torch.zeros_like(sim, dtype=torch.bool)

    idx = torch.arange(B, device=z.device)
    pos_idx = idx[y.bool()]

    pos_mask[pos_idx, B+pos_idx] = True
    pos_mask[B+pos_idx, pos_idx] = True

    # log-softmax across rows
    log_prob = F.log_softmax(sim, dim=1)

    # only keep positive pairs
    loss = -(log_prob * pos_mask).sum() / pos_mask.sum().clamp(min=1)

    return loss

### Precalculations of data

In [ ]:
# Parameters
in_window=2114
out_window=1000
max_jitter=128
genome_file = f"{chrombpnet_dir}/data/hg38.fa"

# Extract all peaks that are either hits (negative) or controls (positive)
all_peaks = []
for key in peakDict.keys():
    all_peaks.extend(peakDict[key]["hits"])
    all_peaks.extend(peakDict[key]["controls"])

# Keep only unique peaks
all_peaks = list(set(all_peaks))

# Get regions and make a DataFrame
chrom = [x.split(":")[0] for x in all_peaks]
start = [x.split(":")[1].split("-")[0] for x in all_peaks]
end = [x.split(":")[1].split("-")[1] for x in all_peaks]
peaks_df = pd.DataFrame({'chr': chrom, 'start': start, 'end': end}, index = all_peaks)

# Get adjusted start and end
peaks_df['mid'] = (peaks_df['start'].astype(int) + peaks_df['end'].astype(int)) // 2
peaks_df['start'] = peaks_df['mid'] - (in_window + 2 * max_jitter) // 2
peaks_df['end'] = peaks_df['mid'] + (in_window + 2 * max_jitter) // 2

# Get sequences
fasta = pysam.FastaFile(genome_file)
seqs = [
    fasta.fetch(chrom, start, end)
    for chrom, start, end in zip(peaks_df.chr, peaks_df.start, peaks_df.end)
]
fasta.close()
peaks_df['sequences'] = seqs

# Get signals for each cell type
for celltype in tqdm(celltypes):
    bw = pyBigWig.open(f"{chrombpnet_dir}/data/{celltype}_coverage_unstranded.bw")
    signals = [
        np.nan_to_num(
            bw.values(chrom, start, end),
            nan=0.0
        )
        for chrom, start, end in zip(peaks_df.chr, peaks_df.start, peaks_df.end)
    ]
    bw.close()
    peaks_df[celltype] = signals

In [ ]:
peaks_df.to_pickle(f"{chrombpnet_dir}/data/celltype_finetune_data.pkl")

### Setup for training

In [ ]:
# Parameters for training
celltypes = ["acinar", "activated_stellate", "alpha", "beta", "cycling", "delta","ductal", "endothelial", "epsilon", "lymphoid", "mast", "myeloid", "pp", "schwann", "quiescent_stellate"]
lr = 1e-5
nbatches_per_epoch = len(celltypes) ** 2 *2
epochs = 10
device = 'cuda'
ema_beta = 0.95
nbatches_burn_in = 1000
eps = 1e-6
nce_weight = 1.0
tau = 0.2

In [ ]:
# Data
peaks_df = pd.read_pickle(f"{chrombpnet_dir}/data/celltype_finetune_data.pkl")

with open(f"{chrombpnet_dir}/data/differential_accessibility.pkl", "rb") as f:
    peakDict = pickle.load(f)

In [ ]:
# Make a dictionary of ChromBPNet models
fold = "0"
model_dict = {}
for celltype in celltypes:
    bias = import_chrombpnet_bias(f"{chrombpnet_dir}/data/fold{fold}/accessibility/{celltype}_bias_model_scaled.h5")
    acc = import_chrombpnet_accessibility(f"{chrombpnet_dir}/data/fold{fold}/accessibility/models/{celltype}.h5", model_name = 'wo_bias')
    model = ChromBPNet(bias, acc, f"{celltype}_{fold}").to(device)
    freeze_layers(model)
    model_dict[celltype] = model

# Make a projection model
projectionModel = ProjectionHead().to(device)

In [ ]:
# Create an optimizer with weights for all models
optimizer = optim.Adam(
    [p for m in model_dict.values() for p in m.parameters()] + list(projectionModel.parameters()),
    lr=lr
)

# Initialize EMA at 1.0
ema_mnll = defaultdict(lambda: 1.0)
ema_mse  = defaultdict(lambda: 1.0)

In [ ]:
# Create stable inits for EMA
scale_mse = defaultdict(list)
scale_mnll = defaultdict(list)

with torch.no_grad():
    for nbatch in tqdm(range(nbatches_burn_in)):
        # Get batch
        labels, sequences, celltypeA_truth, celltypeB_truth, celltypeA, celltypeB = getBatch(celltypes = celltypes, sets = peakDict)

        # Move to device
        sequences = sequences.to(device)
        celltypeA_truth = celltypeA_truth.to(device)
        celltypeB_truth = celltypeB_truth.to(device)
        
        # Forward
        celltypeA_profile, celltypeA_count, celltypeA_embedding = custom_forward(model_dict[celltypeA], sequences)
        celltypeB_profile, celltypeB_count, celltypeB_embedding = custom_forward(model_dict[celltypeB], sequences)
        
        # Losses
        celltypeA_MSE = torch.mean(torch.square(torch.log(celltypeA_truth.sum(dim=-1).reshape(-1, 1)+1) - celltypeA_count), dim=-1).mean()
        celltypeA_MNLL = MNLLLoss(celltypeA_profile, celltypeA_truth).mean()
        celltypeB_MSE = torch.mean(torch.square(torch.log(celltypeB_truth.sum(dim=-1).reshape(-1, 1)+1) - celltypeB_count), dim=-1).mean()
        celltypeB_MNLL = MNLLLoss(celltypeB_profile, celltypeB_truth).mean()

        # Store losses
        scale_mse[celltypeA].append(celltypeA_MSE.item())
        scale_mse[celltypeB].append(celltypeB_MSE.item())
        scale_mnll[celltypeA].append(celltypeA_MNLL.item())
        scale_mnll[celltypeB].append(celltypeB_MNLL.item())

In [ ]:
# Initialize EMAs
for ct in celltypes:
    ema_mnll[ct] = np.mean(scale_mnll[ct])
    ema_mse[ct] = np.mean(scale_mse[ct])

In [ ]:
# Setup loss dictionary
loss_dict = {}
loss_dict['BCE'] = []
loss_dict['MSE'] = {}
loss_dict['MNLL'] = {}
for celltype in celltypes:
    loss_dict['MSE'][celltype] = []
    loss_dict['MNLL'][celltype] = []

# Training loop
for epoch in tqdm(range(epochs)):
    loss_dict_epoch = {}
    loss_dict_epoch['BCE'] = []
    loss_dict_epoch['MSE'] = {}
    loss_dict_epoch['MNLL'] = {}
    for celltype in celltypes:
        loss_dict_epoch['MSE'][celltype] = []
        loss_dict_epoch['MNLL'][celltype] = []

    for nbatch in range(nbatches_per_epoch):
        # Zero grads
        optimizer.zero_grad(set_to_none=True)
        
        # Get data
        labels, sequences, celltypeA_truth, celltypeB_truth, celltypeA, celltypeB = getBatch(celltypes = celltypes, sets = peakDict)
    
        # Move to device
        sequences = sequences.to(device)
        celltypeA_truth = celltypeA_truth.to(device)
        celltypeB_truth = celltypeB_truth.to(device)
        
        # Forward
        celltypeA_profile, celltypeA_count, celltypeA_embedding = custom_forward(model_dict[celltypeA], sequences)
        celltypeA_Z = projectionModel(celltypeA_embedding)
        celltypeB_profile, celltypeB_count, celltypeB_embedding = custom_forward(model_dict[celltypeB], sequences)
        celltypeB_Z = projectionModel(celltypeB_embedding)
        
        # Losses
        celltypeA_MSE = torch.mean(torch.square(torch.log(celltypeA_truth.sum(dim=-1).reshape(-1, 1)+1) - celltypeA_count), dim=-1).mean()
        celltypeA_MNLL = MNLLLoss(celltypeA_profile, celltypeA_truth).mean()
        celltypeB_MSE = torch.mean(torch.square(torch.log(celltypeB_truth.sum(dim=-1).reshape(-1, 1)+1) - celltypeB_count), dim=-1).mean()
        celltypeB_MNLL = MNLLLoss(celltypeB_profile, celltypeB_truth).mean()
        logits = (celltypeA_Z * celltypeB_Z).sum(dim=1) / tau
        BCE = F.binary_cross_entropy_with_logits(logits, labels.float().to(device))
    
        # Update EMAs
        with torch.no_grad():
            ema_mnll[celltypeA] = ema_beta*ema_mnll[celltypeA] + (1-ema_beta)*celltypeA_MNLL.detach().item()
            ema_mse[celltypeA]  = ema_beta*ema_mse[celltypeA]  + (1-ema_beta)*celltypeA_MSE.detach().item()
            ema_mnll[celltypeB] = ema_beta*ema_mnll[celltypeB] + (1-ema_beta)*celltypeB_MNLL.detach().item()
            ema_mse[celltypeB]  = ema_beta*ema_mse[celltypeB]  + (1-ema_beta)*celltypeB_MSE.detach().item()
        
        # Calculate losses scaled by EMA
        celltypeA_loss = celltypeA_MNLL/(ema_mnll[celltypeA]+eps) + celltypeA_MSE/(ema_mse[celltypeA]+eps)
        celltypeB_loss = celltypeB_MNLL/(ema_mnll[celltypeB]+eps) + celltypeB_MSE/(ema_mse[celltypeB]+eps)
    
        # Total loss
        loss_task = 0.5*(celltypeA_loss + celltypeB_loss)
        loss = loss_task + nce_weight * BCE
    
        # Backwards
        loss.backward()
    
        # Step
        optimizer.step()
    
        # Store losses
        loss_dict_epoch['BCE'].append(BCE.item())
        loss_dict_epoch['MNLL'][celltypeA].append((celltypeA_MNLL/(ema_mnll[celltypeA]+eps)).item())
        loss_dict_epoch['MSE'][celltypeA].append((celltypeA_MSE/(ema_mse[celltypeA]+eps)).item())
        loss_dict_epoch['MNLL'][celltypeB].append((celltypeB_MNLL/(ema_mnll[celltypeB]+eps)).item())
        loss_dict_epoch['MSE'][celltypeB].append((celltypeB_MSE/(ema_mse[celltypeB]+eps)).item())

    # Update per epoch losses
    loss_dict['BCE'].append(np.mean(loss_dict_epoch['BCE']))
    for celltype in celltypes:
        loss_dict['MSE'][celltype].append(np.mean(loss_dict_epoch['MSE'][celltype]))
        loss_dict['MNLL'][celltype].append(np.mean(loss_dict_epoch['MNLL'][celltype]))

    print(np.mean(loss_dict_epoch['BCE']))

In [ ]:
# Unfreeze heads
for celltype in celltypes:
    freeze_layers(model_dict[celltype], freeze_output_heads = False)

# Create a new optimizer with weights for all models
optimizer = optim.Adam(
    [p for m in model_dict.values() for p in m.parameters()] + list(projectionModel.parameters()),
    lr=lr
)

In [ ]:
# Training loop
for epoch in tqdm(range(epochs)):
    for nbatch in range(nbatches_per_epoch):
        # Zero grads
        optimizer.zero_grad(set_to_none=True)
        
        # Get data
        labels, sequences, celltypeA_truth, celltypeB_truth, celltypeA, celltypeB = getBatch(celltypes = celltypes, sets = peakDict)
    
        # Move to device
        sequences = sequences.to(device)
        celltypeA_truth = celltypeA_truth.to(device)
        celltypeB_truth = celltypeB_truth.to(device)
        
        # Forward
        celltypeA_profile, celltypeA_count, celltypeA_embedding = custom_forward(model_dict[celltypeA], sequences)
        celltypeA_Z = projectionModel(celltypeA_embedding)
        celltypeB_profile, celltypeB_count, celltypeB_embedding = custom_forward(model_dict[celltypeB], sequences)
        celltypeB_Z = projectionModel(celltypeB_embedding)
        
        # Losses
        celltypeA_MSE = torch.mean(torch.square(torch.log(celltypeA_truth.sum(dim=-1).reshape(-1, 1)+1) - celltypeA_count), dim=-1).mean()
        celltypeA_MNLL = MNLLLoss(celltypeA_profile, celltypeA_truth).mean()
        celltypeB_MSE = torch.mean(torch.square(torch.log(celltypeB_truth.sum(dim=-1).reshape(-1, 1)+1) - celltypeB_count), dim=-1).mean()
        celltypeB_MNLL = MNLLLoss(celltypeB_profile, celltypeB_truth).mean()
        logits = (celltypeA_Z * celltypeB_Z).sum(dim=1) / tau
        BCE = F.binary_cross_entropy_with_logits(logits, labels.float().to(device))
    
        # Update EMAs
        with torch.no_grad():
            ema_mnll[celltypeA] = ema_beta*ema_mnll[celltypeA] + (1-ema_beta)*celltypeA_MNLL.detach().item()
            ema_mse[celltypeA]  = ema_beta*ema_mse[celltypeA]  + (1-ema_beta)*celltypeA_MSE.detach().item()
            ema_mnll[celltypeB] = ema_beta*ema_mnll[celltypeB] + (1-ema_beta)*celltypeB_MNLL.detach().item()
            ema_mse[celltypeB]  = ema_beta*ema_mse[celltypeB]  + (1-ema_beta)*celltypeB_MSE.detach().item()
        
        # Calculate losses scaled by EMA
        celltypeA_loss = celltypeA_MNLL/(ema_mnll[celltypeA]+eps) + celltypeA_MSE/(ema_mse[celltypeA]+eps)
        celltypeB_loss = celltypeB_MNLL/(ema_mnll[celltypeB]+eps) + celltypeB_MSE/(ema_mse[celltypeB]+eps)
    
        # Total loss
        loss_task = 0.5*(celltypeA_loss + celltypeB_loss)
        loss = loss_task + nce_weight * BCE
    
        # Backwards
        loss.backward()
    
        # Step
        optimizer.step()
    
        # Store losses
        loss_dict_epoch['BCE'].append(BCE.item())
        loss_dict_epoch['MNLL'][celltypeA].append((celltypeA_MNLL/(ema_mnll[celltypeA]+eps)).item())
        loss_dict_epoch['MSE'][celltypeA].append((celltypeA_MSE/(ema_mse[celltypeA]+eps)).item())
        loss_dict_epoch['MNLL'][celltypeB].append((celltypeB_MNLL/(ema_mnll[celltypeB]+eps)).item())
        loss_dict_epoch['MSE'][celltypeB].append((celltypeB_MSE/(ema_mse[celltypeB]+eps)).item())

    # Update per epoch losses
    loss_dict['BCE'].append(np.mean(loss_dict_epoch['BCE']))
    for celltype in celltypes:
        loss_dict['MSE'][celltype].append(np.mean(loss_dict_epoch['MSE'][celltype]))
        loss_dict['MNLL'][celltype].append(np.mean(loss_dict_epoch['MNLL'][celltype]))

    print(np.mean(loss_dict_epoch['BCE']))

In [ ]:
import seaborn as sns

In [ ]:
mse_df = pd.DataFrame(loss_dict['MSE'])
mse_df["Epoch"] = mse_df.index
mse_df = mse_df.melt(id_vars="Epoch", var_name="Celltype", value_name="MSE")
ax = sns.lineplot(data=mse_df, x="Epoch", y="MSE", hue="Celltype")
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()

In [ ]:
mnll_df = pd.DataFrame(loss_dict['MNLL'])
mnll_df["Epoch"] = mnll_df.index
mnll_df = mnll_df.melt(id_vars="Epoch", var_name="Celltype", value_name="MNLL")
ax = sns.lineplot(data=mnll_df, x="Epoch", y="MNLL", hue="Celltype")
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()

In [ ]:
nce_df = pd.DataFrame({'BCE': loss_dict['BCE']})
nce_df["Epoch"] = nce_df.index
sns.lineplot(data=nce_df, x="Epoch", y="BCE")

In [ ]:
# Save weights
for celltype in celltypes:
    torch.save(model_dict[celltype].state_dict(), f"{chrombpnet_dir}/data/celltype_models/{celltype}.pt")

### Motifs

In [ ]:
window_size = 500
import os
import pickle
genome_file = f"{chrombpnet_dir}/data/hg38.fa"
motif_path = f"{chrombpnet_dir}/data/motifs.meme"


In [ ]:
# Load all peaks
peaks = pd.read_csv(f"{chrombpnet_dir}/data/peaks_no_blacklist_summits.bed", sep="\t", header=None)

# Onehot encode
hg38_ambiguity_letters = ["R", "Y", "S", "W", "K", "M", "B", "D", "H", "V", "N"]
X = extract_loci(peaks[[0,1,2]], genome_file, ignore=hg38_ambiguity_letters)

In [ ]:
# Find motifs
hits = fimo(motif_path, X[:,:,(1057 - (window_size//2)):(1057 + (window_size//2))])

# Output
with open(f"{chrombpnet_dir}/data/all_peaks.motifs", "wb") as f:
    pickle.dump(hits, f)

### Ablation

In [ ]:
# Setup
batch_size = 512
celltypes = ["acinar", "activated_stellate", "alpha", "beta", "cycling", "delta","ductal", "endothelial", "epsilon", "lymphoid", "mast", "myeloid", "pp", "schwann", "quiescent_stellate"]
folds = [0,1,2,3,4]
n_resamples = 5
window_size = 500
Nsample = 1000
maxShuffle = 10000
output_path = f"{chrombpnet_dir}/data/motif_ablation/"
genome_file = f"{chrombpnet_dir}/data/hg38.fa"

In [ ]:
# Load all peaks
peaks = pd.read_csv(f"{chrombpnet_dir}/data/peaks_no_blacklist_summits.bed", sep="\t", header=None)

# Onehot encode
hg38_ambiguity_letters = ["R", "Y", "S", "W", "K", "M", "B", "D", "H", "V", "N"]
X = extract_loci(peaks[[0,1,2]], genome_file, ignore=hg38_ambiguity_letters)

In [ ]:
# Load motifs
with open(f"{chrombpnet_dir}/data/all_peaks.motifs", "rb") as f:
    hits = pickle.load(f)

In [ ]:
# Filter motifs
keep = []
for midx in range(len(hits)):
    nhits = len(set(hits[midx]['sequence_name'].tolist())) / len(peaks)
    if nhits >= 0.0001:
        if nhits <= 0.3:
            keep.append(midx)

hits_filtered = [hits[i] for i in keep]

In [ ]:
# Create results dictionary
result_dict = {}
for celltype in celltypes:
    result_dict[celltype] = {}
    for midx in range(len(hits)):
        result_dict[celltype][midx] = {}
        for fold in folds:
            result_dict[celltype][midx][fold] = { 'before': np.array([]), 'after': np.array([])}

In [ ]:
# Create a dictionary of peaks
peak_dict = {}
for celltype in celltypes:
    peaks_path = f"{chrombpnet_dir}/data/peaks/{celltype}_filtered.bed"
    celltype_peaks = pd.read_csv(peaks_path, sep="\t", header = None)
    peak_dict[celltype] = celltype_peaks

In [ ]:
# Set models in eval mode
for m in model_dict.values():
    m.eval()

In [ ]:
def shuffle_sequences(X, start=0, end=-1, n=1, random_state=None):
	if not isinstance(random_state, np.random.RandomState):
		random_state = np.random.RandomState(random_state)

	X_shufs = []
	for i in range(n):
		idxs = np.arange(end-start)
		random_state.shuffle(idxs)

		X_ = torch.clone(X)
		X_[:, :, start:end] = X[:, :, start:end][:, :, idxs]
		X_shufs.append(X_)

	return torch.stack(X_shufs).permute(1, 0, 2, 3)


In [ ]:
fold = 0
with torch.no_grad():
    # Load train test validation split
    with open(f"{chrombpnet_dir}/data/fold_{fold}.json", "r") as f:
        fold_split = json.load(f)
    
    # Get indexes of peaks on test chromosomes
    ids = np.argwhere(peaks[0].isin(fold_split['test']))[:,0]
    
    # Subset peaks and reset index (to match order in X_fold)
    peaks_fold = peaks[peaks.index.isin(ids)].copy()
    
    # Loop over motifs
    for midx in tqdm(range(len(hits_filtered))):
        # Extract motif hits
        selectedMotif = hits[midx]
        selectedMotif = selectedMotif[selectedMotif['sequence_name'].isin(peaks_fold.index)].copy()
        if len(selectedMotif) > maxShuffle:
            selectedMotif = selectedMotif.sample(n = maxShuffle)        
    
        if len(selectedMotif) > 0:
            # Get all original sequences with hits
            X_original = X[np.array(selectedMotif['sequence_name'].tolist()),:,:]
    
            # Perturb all original sequences
            X_perturb = torch.empty(len(X_original), n_resamples, 4, 2114)
            selectedMotif['start'] = selectedMotif['start'] + (1057 - (window_size//2))
            selectedMotif['end'] = selectedMotif['end'] + (1057 - (window_size//2))
            annotation_tensor = torch.tensor(np.array(selectedMotif[['sequence_name', 'start', 'end']]))
            for i, (idx, start, end) in enumerate(annotation_tensor):
                X_perturb[i,:,:,:] = shuffle_sequences(X_original[i,:,:].unsqueeze(0), start=start, end=end, n=n_resamples, random_state=None)
    
            # Loop over cell types
            for celltype in celltypes:
                # Load and process peaks
                celltype_peaks = peak_dict[celltype]
                celltype_peaks_fold = celltype_peaks[celltype_peaks[0].isin(fold_split['test'])].copy()
                celltype_ids = np.argwhere(peaks[3].isin(celltype_peaks_fold[3]))[:,0]
                celltype_hits = np.argwhere(selectedMotif['sequence_name'].isin(celltype_ids))[:,0]
        
                # Only proceed if there are hits in peaks
                if len(celltype_hits) > 0:
                    # Get subset
                    X_original_celltype = X_original[celltype_hits,:,:]
                    X_perturb_celltype = X_perturb[celltype_hits,:,:,:]
    
                    # Subsampling
                    N = len(X_original_celltype)
                    if N > Nsample:
                        idx = np.random.choice(N, size=Nsample, replace=False)
                        X_original_celltype = X_original_celltype[idx,:,:]
                        X_perturb_celltype = X_perturb_celltype[idx,:,:,:]
    
                    # Predict for the perturbed sequences
                    N, R, C, L = X_perturb_celltype.shape
                    X_flat = X_perturb_celltype.permute(1, 0, 2, 3).reshape(R * N, C, L)
                    preds = []
                    for i in range(0, R * N, batch_size):
                        batch = X_flat[i:i + batch_size].to('cuda', non_blocking=True)
                        _, pred, _ = custom_forward(model_dict[celltype], batch)
                        preds.append(pred)
                    preds_flat = torch.cat(preds, dim=0)
                    out_shape = preds_flat.shape[1:]
                    preds_RN = preds_flat.reshape(R, N, *out_shape)
                    result_dict[celltype][midx][fold]['after'] = preds_RN.squeeze()
    
                    # Predict for original sequences and store
                    before_fold = []
                    for i in range(0, len(X_original_celltype), batch_size):
                        end = i+batch_size
                        if end > len(X_original_celltype):
                            end = len(X_original_celltype)
                        batch = X_original_celltype[i : end,:,:].to('cuda')
                        _, pred, _ = custom_forward(model_dict[celltype], batch.to(torch.float32))
                        before_fold.append(pred)
                    result_dict[celltype][midx][fold]['before'] = torch.cat(before_fold, dim=0).cpu().numpy().squeeze()
    
    # Output after each fold
    with open(f"{chrombpnet_dir}/data/motif_activity.pkl", "wb") as f:
        pickle.dump(result_dict, f)

In [ ]:
import io 
class CPU_Unpickler(pickle.Unpickler):
    def find_class(self, module, name):
        # This is what pickle uses to rebuild torch storages
        if module == "torch.storage" and name == "_load_from_bytes":
            return lambda b: torch.load(io.BytesIO(b), map_location="cpu")
        return super().find_class(module, name)

path = f"{chrombpnet_dir}/data/motif_activity.pkl"
with open(path, "rb") as f:
    data = CPU_Unpickler(f).load()

In [ ]:
# Load motifs
with open(f"{chrombpnet_dir}/data/all_peaks.motifs", "rb") as f:
    hits = pickle.load(f)

In [ ]:
# Load all peaks
peaks = pd.read_csv(f"{chrombpnet_dir}/data/peaks_no_blacklist_summits.bed", sep="\t", header=None)

# Filter motifs
keep = []
for midx in range(len(hits)):
    nhits = len(set(hits[midx]['sequence_name'].tolist())) / len(peaks)
    if nhits >= 0.0001:
        if nhits <= 0.3:
            keep.append(midx)

hits_filtered = [hits[i] for i in keep]

In [ ]:
for midx in range(len(hits_filtered)):
    ma = []
    for celltype in celltypes:
        if len(data[celltype][midx][0]['after']) > 0:
            ma.append((data[celltype][midx][0]['before'] - data[celltype][midx][0]['after'].cpu().numpy()).mean())
        else:
            ma.append(np.nan)

    # Structure the data
    tmp_df = pd.DataFrame(np.array(ma), index = celltypes)
    tmp_df = tmp_df.rename(columns = {0: hits_filtered[midx]['motif_name'][0].split(" ")[0]}).T

    # Concat
    if midx == 0:
        ma_df = tmp_df
    else:
        ma_df = pd.concat([ma_df, tmp_df])

In [ ]:
ma_df = ma_df.dropna()

In [ ]:
ma_filtered = ma_df.copy()
#ma_filtered = ma_filtered[~ma_filtered.index.str.contains("ZNF")]
#ma_filtered = ma_filtered[~ma_filtered.index.str.contains("C2H2")]


In [ ]:
Z = ((ma_filtered.T - ma_filtered.T.mean()) / ma_filtered.T.std(ddof=0)).T

In [ ]:
winners = np.argmax(Z, axis = 1)

selected = []
for celltype in celltypes:
    selected.extend(Z.iloc[np.argwhere(winners == np.argwhere(Z.columns.isin([celltype]))[0])[:,0],:][celltype].nlargest(5).index.tolist())

## Attribution in highly variable sites

In [ ]:
# Load all peaks
peaks = pd.read_csv(f"{chrombpnet_dir}/data/peaks_no_blacklist_summits.bed", sep="\t", header=None)
genome_file = f"{chrombpnet_dir}/data/hg38.fa"
device = 'cuda'
batch_size = 128

In [ ]:
# Onehot encode
hg38_ambiguity_letters = ["R", "Y", "S", "W", "K", "M", "B", "D", "H", "V", "N"]
X = extract_loci(peaks[[0,1,2]], genome_file, ignore=hg38_ambiguity_letters)

# Define cell types
celltypes = ["acinar", "activated_stellate", "alpha", "beta", "delta","ductal", "endothelial", "epsilon", "lymphoid", "mast", "myeloid", "pp", "schwann", "quiescent_stellate"]

In [ ]:
# Make a dictionary of ChromBPNet models
fold = "0"
model_dict = {}
for celltype in celltypes:
    bias = import_chrombpnet_bias(f"{chrombpnet_dir}/data/fold{fold}/accessibility/{celltype}_bias_model_scaled.h5")
    acc = import_chrombpnet_accessibility(f"{chrombpnet_dir}/data/fold{fold}/accessibility/models/{celltype}.h5", model_name = 'wo_bias')
    model = ChromBPNet(bias, acc, f"{celltype}_{fold}").to(device)
    model_dict[celltype] = model
    state = torch.load(f"{chrombpnet_dir}/data/celltype_models/{celltype}.pt")
    model_dict[celltype].load_state_dict(state)
    model_dict[celltype].eval()

In [ ]:
allPeaksPredictionDict = {}
for celltype in celltypes:
    allPeaksPredictionDict[celltype] = []

In [ ]:
with torch.no_grad():
    for i in tqdm(range(0, len(X), batch_size)):
        end = i+batch_size
        if end > len(X):
            end = len(X)
        batch = X[i : end,:,:].to('cuda').to(torch.float32)
        for celltype in celltypes:
            _, pred, _ = custom_forward(model_dict[celltype], batch)
            allPeaksPredictionDict[celltype].append(pred.cpu().numpy())

In [ ]:
# Stack per celltype
for celltype in celltypes:
    allPeaksPredictionDict[celltype] = np.vstack(allPeaksPredictionDict[celltype])

In [ ]:
# Z scaling
Z = {}
for celltype in celltypes:
    Z[celltype] = (allPeaksPredictionDict[celltype] - allPeaksPredictionDict[celltype].mean()) / allPeaksPredictionDict[celltype].std() 
    Z[celltype] = Z[celltype].squeeze()

allPeaks = pd.DataFrame(Z)

In [ ]:
winner_peaks = []
for celltype in celltypes:
    winners = allPeaks.iloc[np.argwhere(np.argmax(allPeaks, axis = 1) == np.argwhere(allPeaks.columns.isin([celltype]))[0])[:,0],:]
    winners = winners[winners[celltype] >= 0.5]
    winner_peaks.extend((winners[celltype] / winners.drop(columns=celltype).max(axis = 1)).nlargest(1000).index.tolist())

In [ ]:
# Get peak winners and OHE
peak_winners = peaks[peaks.index.isin(winner_peaks)].copy()
peak_winners.to_csv(f"{chrombpnet_dir}/MarginPeaks_top1000.csv")

In [ ]:
from bpnetlite.bpnet import CountWrapper
from tangermeme.io import read_meme
from tangermeme.annotate import annotate_seqlets
from tangermeme.seqlet import recursive_seqlets
from tangermeme.annotate import count_annotations
from bpnetlite.bpnet import CountWrapper, ControlWrapper
from qvalue.qvalue import qvalue, pi0est
from captum.attr import DeepLiftShap
from tangermeme.deep_lift_shap import deep_lift_shap
from tangermeme.deep_lift_shap import hypothetical_attributions

In [ ]:
# Process peaks
hg38_ambiguity_letters = ["R", "Y", "S", "W", "K", "M", "B", "D", "H", "V", "N"]
genome_file = f"{chrombpnet_dir}/data/hg38.fa"
peak_winners = pd.read_csv(f"{chrombpnet_dir}/MarginPeaks_top1000.csv", index_col=0)
X = extract_loci(peak_winners[['0','1','2']], genome_file, ignore=hg38_ambiguity_letters).to(torch.float64)
X = X[X.sum(axis=1).sum(axis=1) == 2114,:,:]

# Define cell types
celltypes = ["acinar", "activated_stellate", "alpha", "beta", "delta","ductal", "endothelial", "epsilon", "lymphoid", "mast", "myeloid", "pp", "schwann", "quiescent_stellate"]

In [ ]:
from tangermeme.ersatz import dinucleotide_shuffle
X_ref = dinucleotide_shuffle(X, n=10, random_state=0).to(torch.float64)

In [ ]:
X = X.to('cuda').to(torch.float64)
X_ref = X_ref.to('cuda').to(torch.float64)

In [ ]:
motif_path = f"{chrombpnet_dir}/data/motifs.meme"
motifs = read_meme(motif_path)
motif_names = np.array([name.split(' ')[0].split(":")[0] for name in motifs.keys()])

In [ ]:
import torch
import torch.nn.functional as F
import warnings
from tqdm import trange

def hypothetical_attributions_fast(multipliers, X, references):
    m = multipliers[0]   # (B, 4, L)
    r = references[0]    # (B, 4, L)
    base = (r * m).sum(dim=1, keepdim=True)   # (B, 1, L)
    projected = m - base                       # (B, 4, L)
    return (projected.to(dtype=X[0].dtype),)

def _register_hooks_force(module):
    # Register even if hooks exist; store our handles so we can remove them later.
    if not isinstance(module, tuple(module._NON_LINEAR_OPS.keys())):
        return
    module.handles = getattr(module, "handles", [])
    module.handles.append(module.register_forward_hook(_f_hook))
    module.handles.append(module.register_forward_pre_hook(_fp_hook))
    module.handles.append(module.register_full_backward_hook(_b_hook))

def _clear_hooks_force(module):
    if hasattr(module, "handles"):
        for h in module.handles:
            h.remove()
        del module.handles

def deep_lift_shap_fast(
    model, X, target=0, batch_size=32, references=None,
    warning_threshold=0.001, print_convergence_deltas=False,
    calculate_convergence=False, verbose=False,
    hypothetical=False,  # keep same meaning as original
):
    # references must be tensor (N,S,4,L)
    n_shuffles = references.shape[1]

    _NON_LINEAR_OPS = {
        torch.nn.ReLU: _nonlinear,
        torch.nn.ReLU6: _nonlinear,
        torch.nn.RReLU: _nonlinear,
        torch.nn.SELU: _nonlinear,
        torch.nn.CELU: _nonlinear,
        torch.nn.GELU: _nonlinear,
        torch.nn.SiLU: _nonlinear,
        torch.nn.Mish: _nonlinear,
        torch.nn.GLU: _nonlinear,
        torch.nn.ELU: _nonlinear,
        torch.nn.LeakyReLU: _nonlinear,
        torch.nn.Sigmoid: _nonlinear,
        torch.nn.Tanh: _nonlinear,
        torch.nn.Softplus: _nonlinear,
        torch.nn.Softshrink: _nonlinear,
        torch.nn.LogSigmoid: _nonlinear,
        torch.nn.PReLU: _nonlinear,
        torch.nn.MaxPool1d: _maxpool,
        torch.nn.MaxPool2d: _maxpool,
        torch.nn.Softmax: _softmax
    }

    for module in model.modules():
        module._NON_LINEAR_OPS = _NON_LINEAR_OPS

    try:
        model.apply(_register_hooks_force)
    except Exception as e:
        model.apply(_clear_hooks_force)
        raise e

    attributions, Xi, rj, attr_ = [], [], [], []
    n, z = X.shape[0] * n_shuffles, 0

    for i in trange(n, disable=not verbose):
        Xi.append(i // n_shuffles)
        rj.append(i % n_shuffles)

        if len(Xi) == batch_size or i == (n - 1):
            _X = X[Xi].detach().requires_grad_(True)
            _ref = references[Xi, rj].detach()  # no grad needed

            try:
                Xcat = torch.cat([_X, _ref], dim=0)
                y = model(Xcat)[:, target]

                multipliers = torch.autograd.grad(y.sum(), _X)[0]  # DeepLIFT multipliers if hooks are active

                if calculate_convergence:
                    output_diff = torch.sub(*torch.chunk(y, 2))
                    input_diff = torch.sum((_X - _ref) * multipliers, dim=(1, 2))
                    deltas = (output_diff - input_diff).abs()

                    if torch.any(deltas > warning_threshold):
                        warnings.warn(f"Convergence deltas too high: {deltas}", RuntimeWarning)
                    if print_convergence_deltas:
                        print(deltas)

            except Exception as e:
                model.apply(_clear_hooks_force)
                raise e

            # IMPORTANT: match original post-processing
            multipliers = hypothetical_attributions_fast((multipliers,), (_X,), (_ref,))[0]  # (B,4,L)

            attr_.extend(list(multipliers.detach().cpu()))

            while len(attr_) >= n_shuffles:
                chunk = torch.stack(attr_[:n_shuffles]).mean(dim=0)  # (4,L)
                if not hypothetical:
                    chunk = chunk * X[z].detach().cpu()
                attributions.append(chunk)
                attr_ = attr_[n_shuffles:]
                z += 1

            Xi, rj = [], []

    model.apply(_clear_hooks_force)
    for module in model.modules():
        del module._NON_LINEAR_OPS

    return torch.stack(attributions)

def _nonlinear(module, grad_input, grad_output):
	delta_in_ = torch.sub(*module.input.chunk(2))
	delta_out_ = torch.sub(*module.output.chunk(2))

	delta_in = torch.cat([delta_in_, delta_in_])
	delta_out = torch.cat([delta_out_, delta_out_])

	delta = delta_out / delta_in
	idxs = torch.abs(delta_in) < 1e-6

	return (torch.where(idxs, grad_input[0], grad_output[0] * delta),)

def _softmax(module, grad_input, grad_output):
	delta_in_ = torch.sub(*module.input.chunk(2))
	delta_out_ = torch.sub(*module.output.chunk(2))

	delta_in = torch.cat([delta_in_, delta_in_])
	delta_out = torch.cat([delta_out_, delta_out_])

	delta = delta_out / delta_in
	idxs = torch.abs(delta_in) < 1e-6

	grad_input_unnorm = torch.where(idxs, grad_input[0], grad_output[0] * delta)

	n = grad_input[0].numel()
	new_grad_inp = grad_input_unnorm - grad_input_unnorm.sum() * 1 / n
	return (new_grad_inp,)


def _maxpool(module, grad_input, grad_output):
	if isinstance(module, torch.nn.MaxPool1d):
		pool_func, unpool_func = F.max_pool1d, F.max_unpool1d
	elif isinstance(module, torch.nn.MaxPool2d):
		pool_func, unpool_func = F.max_pool2d, F.max_unpool2d
	else:
		raise ValueError("module must be either MaxPool1d or MaxPool2d")


	with torch.no_grad():
		delta_in_ = torch.sub(*module.input.chunk(2))
		delta_in = torch.cat([delta_in_, delta_in_])

		output, output_ref = module.output.chunk(2)
		delta_out_xmax = torch.max(output, output_ref)
		delta_out = torch.cat([delta_out_xmax - output_ref, 
			output - delta_out_xmax])

		_, indices = pool_func(module.input, module.kernel_size, module.stride, 
			module.padding, module.dilation, module.ceil_mode, True)

		unpool_ = unpool_func(grad_output[0] * delta_out, indices, 
			module.kernel_size, module.stride, module.padding, 
			list(module.input.shape))
		unpool_delta, unpool_ref_delta = torch.chunk(unpool_, 2)

	unpool_delta_ = unpool_delta + unpool_ref_delta
	unpool_delta = torch.cat([unpool_delta_, unpool_delta_])
	idxs = torch.abs(delta_in) < 1e-7

	new_grad_inp = torch.where(idxs, grad_input[0], unpool_delta / delta_in)
	return (new_grad_inp,)

def _register_hooks(module): 
	if len(module._backward_hooks) > 0:
		return
	if not isinstance(module, tuple(module._NON_LINEAR_OPS.keys())):
		return

	module.handles = []
	module.handles.append(module.register_forward_hook(_f_hook))
	module.handles.append(module.register_forward_pre_hook(_fp_hook))
	module.handles.append(module.register_full_backward_hook(_b_hook))


def _clear_hooks(module):
	if hasattr(module, "handles") and len(module.handles) > 0:
		for handle in module.handles:
			handle.remove()

		del module.handles


def _fp_hook(module, inputs): 
	module.input = inputs[0].clone().detach()


def _f_hook(module, inputs, outputs):
	module.output = outputs.clone().detach()


def _b_hook(module, grad_input, grad_output):
	return module._NON_LINEAR_OPS[type(module)](module, grad_input, 
		grad_output)

In [ ]:
fold = 0
celltype = "beta"
bias = import_chrombpnet_bias(f"{chrombpnet_dir}/data/fold{fold}/accessibility/{celltype}_bias_model_scaled.h5")
acc = import_chrombpnet_accessibility(f"{chrombpnet_dir}/data/fold{fold}/accessibility/models/{celltype}.h5", model_name = 'wo_bias')
model = ChromBPNet(bias, acc, f"{celltype}_{fold}")
state = torch.load(f"{chrombpnet_dir}/data/celltype_models/{celltype}.pt")
model.load_state_dict(state)
wrapped = CountWrapper(ControlWrapper(model)).to('cuda').to(torch.float64)

In [ ]:
X_test = X[:32,:,:]
X_ref_test = X_ref[:32,:,:]

In [ ]:
%%time
deep_lift_shap_fast(wrapped, X_test, references=X_ref_test, calculate_convergence=False, verbose = True, batch_size=256)

In [ ]:
fold = 0
celltype = "beta"
bias = import_chrombpnet_bias(f"{chrombpnet_dir}/data/fold{fold}/accessibility/{celltype}_bias_model_scaled.h5")
acc = import_chrombpnet_accessibility(f"{chrombpnet_dir}/data/fold{fold}/accessibility/models/{celltype}.h5", model_name = 'wo_bias')
model = ChromBPNet(bias, acc, f"{celltype}_{fold}")
state = torch.load(f"{chrombpnet_dir}/data/celltype_models/{celltype}.pt")
model.load_state_dict(state)
wrapped = CountWrapper(ControlWrapper(model)).to('cpu')

In [ ]:
X_test = X[:32,:,:].to(torch.float32).cpu()
X_ref_test = X_ref[:32,:,:].to(torch.float32).cpu()

In [ ]:
%%time
deep_lift_shap(wrapped, X_test, references=X_ref_test, device='cpu', verbose = True)

In [ ]:
nnear = len(motifs)
qval_threshold = 0.1
seqlets = []
fold = 0
for celltype in celltypes:
    # Wrap the model
    bias = import_chrombpnet_bias(f"{chrombpnet_dir}/data/fold{fold}/accessibility/{celltype}_bias_model_scaled.h5")
    acc = import_chrombpnet_accessibility(f"{chrombpnet_dir}/data/fold{fold}/accessibility/models/{celltype}.h5", model_name = 'wo_bias')
    model = ChromBPNet(bias, acc, f"{celltype}_{fold}")
    state = torch.load(f"{chrombpnet_dir}/data/celltype_models/{celltype}.pt")
    model.load_state_dict(state)
    wrapped = CountWrapper(ControlWrapper(model)).to('cuda').to(torch.float64)
    
    # Calculate attributions
    X_attr = deep_lift_shap_fast(wrapped, X, references=X_ref, calculate_convergence=False, verbose = True)
    
    # Find seqlets
    X_seqlet = recursive_seqlets(X_attr.sum(dim=1).to(torch.float32), threshold=0.001, additional_flanks=2)
    seqlets.append(X_seqlet)
    
    # Annotate seqlets using motifs
    idxs, pvals = annotate_seqlets(X, X_seqlet, motifs, n_nearest = nnear)
    q, pi0 = qvalue(pvals.reshape(pvals.shape[0] * pvals.shape[1]).numpy())
    q = q.reshape(pvals.shape)
    
    # Count annoations
    counts_ = torch.zeros((len(X), len(motif_names)))
    for near in range(nnear):
        peak = X_seqlet['example_idx'].copy()
        motif = idxs[:, near]
        qval = q[:, near]
        if (qval <= qval_threshold).sum() > 0:
            seqlets = (peak[(qval <= qval_threshold).cpu().numpy()], motif[qval <= qval_threshold])
            counts_ += count_annotations(seqlets, shape=(len(X), len(motif_names))).float()
    
    # Store counts
    count_df_tmp = pd.DataFrame({celltype: counts_.sum(axis = 0).numpy()}, index = motif_names)

    # Concat
    if celltype == celltypes[0]:
        counts_df = count_df_tmp.copy()
    else:
        counts_df = pd.merge(counts_df, count_df_tmp, left_index=True, right_index=True)

In [ ]:
idxs, pvals = annotate_seqlets(X, X_seqlet, motifs, n_nearest = nnear)

In [ ]:
counts_norm = counts_norm[counts_norm.sum(axis = 1) > 0]

In [ ]:
counts_Z = ((counts_norm.T - counts_norm.mean(axis = 1)) / counts_norm.std(axis = 1)).T

### Marginization

In [ ]:
celltypes = ["acinar", "activated_stellate", "alpha", "beta", "cycling", "delta","ductal", "endothelial", "epsilon", "lymphoid", "mast", "myeloid", "pp", "schwann", "quiescent_stellate"]
device = 'cuda'
batch_size = 64
n_sequences = 500
n_shuffles = 5
peaks_df = pd.read_pickle(f"{chrombpnet_dir}/data/celltype_finetune_data.pkl")

In [ ]:
# Make a dictionary of ChromBPNet models
fold = "0"
model_dict = {}
for celltype in celltypes:
    bias = import_chrombpnet_bias(f"{chrombpnet_dir}/data/fold{fold}/accessibility/{celltype}_bias_model_scaled.h5")
    acc = import_chrombpnet_accessibility(f"{chrombpnet_dir}/data/fold{fold}/accessibility/models/{celltype}.h5", model_name = 'wo_bias')
    model = ChromBPNet(bias, acc, f"{celltype}_{fold}").to(device)
    model_dict[celltype] = model
    state = torch.load(f"{chrombpnet_dir}/data/celltype_models/{celltype}.pt")
    model_dict[celltype].load_state_dict(state)
    model_dict[celltype].eval()

In [ ]:
def sample_from_pwm(pwm, N=1, T=1.0, BASES = np.array(["A", "C", "G", "T"]), onehot = True):
    # Temperature scaling
    p = pwm ** (1.0 / T)
    p = p / p.sum(axis=1, keepdims=True)

    L = p.shape[0]
    
    # Sample indices: shape (N, L)
    idx = np.array([
        np.random.choice(4, size=N, p=p[j])
        for j in range(L)
    ]).T  # transpose to (N, L)

    # Convert to sequences
    seqs = ["".join(BASES[row]) for row in idx]
    if onehot:
        seqs = [one_hot_encode(s) for s in seqs]
        seqs = torch.stack(seqs)
    
    return seqs

In [ ]:
# Get randomly selected real peaks and dinucleotide shuffle
from tangermeme.ersatz import dinucleotide_shuffle
random_sequences = peaks_df.sample(n = n_sequences, random_state=0)['sequences'].tolist()
random_sequences = [one_hot_encode(s[128:2242].upper(), ignore = ["R", "Y", "S", "W", "K", "M", "B", "D", "H", "V", "N"]) for s in random_sequences]
X_shuf = [dinucleotide_shuffle(s.unsqueeze(dim= 0), n=n_shuffles, random_state=0) for s in random_sequences]
X_shuf = torch.hstack(X_shuf).squeeze(dim= 0)

In [ ]:
baseDict = {}
with torch.no_grad():
    for celltype in tqdm(celltypes):
        before = []
        for i in range(0, len(X_shuf), batch_size):
            end = i+batch_size
            if end > len(X_shuf):
                end = len(X_shuf)
            batch = X_shuf[i : end,:,:].to('cuda')
            _, pred, _ = custom_forward(model_dict[celltype], batch.to(torch.float32))
            before.append(pred)
        baseDict[celltype] = torch.vstack(before).cpu().numpy()

In [ ]:
from tangermeme.deep_lift_shap import deep_lift_shap
from bpnetlite.bpnet import ProfileWrapper

In [ ]:
wrap = CountWrapper(model_dict['beta'])

In [ ]:
from tangermeme.utils import random_one_hot
from tangermeme.ersatz import substitute

X = random_one_hot((1, 4, 2114)).type(torch.float32)
X = substitute(X, "GTGACTCATC")

In [ ]:
from matplotlib import pyplot as plt
import seaborn; seaborn.set_style('whitegrid')
from tangermeme.plot import plot_logo

plt.figure(figsize=(10, 2))
ax = plt.subplot(111)
plot_logo(X_attr[0, :, 950:1050], ax=ax)

plt.xlabel("Genomic Coordinate")
plt.ylabel("Attributions")
plt.title("DeepLIFT Attributions for GM12878 JunD")
plt.ylim(-0.05, 0.35)
plt.show()

In [ ]:
from memelite.io import read_meme
motif_path = f"{chrombpnet_dir}/data/motifs.meme"
motifs = read_meme(motif_path)

In [ ]:
with torch.no_grad():
    for motif in tqdm(motifs.keys()):
        # Sample motifs
        sampled_motifs = sample_from_pwm(motifs[motif].T, T = 1.0, N = len(X_shuf))
        
        # Insert motifs
        X = sampled_motifs.shape[2]
        start = (X_shuf.shape[2] - X) // 2
        end = start + X
        X_shuf_inserted = X_shuf.clone()
        X_shuf_inserted[:, :, start:end] = sampled_motifs
    
        # Create dict to hold predictions
        after = {}
        for celltype in celltypes:
            after[celltype] = []
        
        # Loop over sequences
        for i in range(0, len(X_shuf_inserted), batch_size):
            # Get batch
            end = i+batch_size
            if end > len(X_shuf_inserted):
                end = len(X_shuf_inserted)
            batch = X_shuf_inserted[i : end,:,:].to('cuda')
        
            # Loop over celltypes
            for celltype in celltypes:
                _, pred, _ = custom_forward(model_dict[celltype], batch.to(torch.float32))
                after[celltype].append(pred.cpu().numpy())
    
        # Stack per celltype
        means = []
        for celltype in celltypes:
            after[celltype] = np.vstack(after[celltype])
            means.append((after[celltype] - baseDict[celltype]).mean())
        
        # Setup as df
        tmp_df = pd.DataFrame({motif: means}, index = celltypes).T
        
        # Concatenate
        if motif == list(motifs.keys())[0]:
            margin_df = tmp_df
        else:
            margin_df = pd.concat([margin_df, tmp_df])

In [ ]:
ma_filtered = margin_df.copy()
ma_filtered = ma_filtered[~ma_filtered.index.str.contains("::")]
#ma_filtered = ma_filtered[~ma_filtered.index.str.contains("ZNF")]
#ma_filtered = ma_filtered[~ma_filtered.index.str.contains("C2H2")]
ma_filtered

In [ ]:
without_cycling = list(set(celltypes) - set(['cycling']))
Z = ((ma_filtered.T - ma_filtered.T.mean()) / ma_filtered.T.std()).T
Z = Z.drop(columns=["cycling"])
winners = np.argmax(Z, axis = 1)
allwinners = []
for celltype in without_cycling:
    winner_motifs = Z.iloc[np.argwhere(winners == np.argwhere(Z.columns.isin([celltype]))[0])[:,0],:].copy()
    winner_motifs = winner_motifs[winner_motifs[celltype] >= 0.5]
    allwinners.extend((winner_motifs[celltype] - winner_motifs.drop(columns=celltype).max(axis = 1)).nlargest(5).index.tolist())

In [ ]:
celltype = "schwann"
winner_motifs = Z.iloc[np.argwhere(winners == np.argwhere(Z.columns.isin([celltype]))[0])[:,0],:].copy()
winner_motifs = winner_motifs[winner_motifs[celltype] >= 0.5]
(winner_motifs[celltype] - winner_motifs.drop(columns=celltype).max(axis = 1)).index.tolist()

### Ablation in highly variable sits

In [ ]:
# Load all peaks
peaks = pd.read_csv(f"{chrombpnet_dir}/data/peaks_no_blacklist_summits.bed", sep="\t", header=None)
genome_file = f"{chrombpnet_dir}/data/hg38.fa"

# Onehot encode
hg38_ambiguity_letters = ["R", "Y", "S", "W", "K", "M", "B", "D", "H", "V", "N"]
X = extract_loci(peaks[[0,1,2]], genome_file, ignore=hg38_ambiguity_letters)

In [ ]:
allPeaksPredictionDict = {}
with torch.no_grad():
    for celltype in tqdm(celltypes):
        before = []
        for i in tqdm(range(0, len(X), batch_size)):
            end = i+batch_size
            if end > len(X):
                end = len(X)
            batch = X[i : end,:,:].to('cuda')
            _, pred, _ = custom_forward(model_dict[celltype], batch.to(torch.float32))
            before.append(pred)
        allPeaksPredictionDict[celltype] = torch.vstack(before).cpu().numpy()

In [ ]:
# Stack per celltype
for celltype in celltypes:
    allPeaksPredictionDict[celltype] = np.vstack(allPeaksPredictionDict[celltype])

In [ ]:
Z = {}
for celltype in celltypes:
    Z[celltype] = (allPeaksPredictionDict[celltype] - allPeaksPredictionDict[celltype].mean()) / allPeaksPredictionDict[celltype].std() 

In [ ]:
import gc
allPeaks = pd.DataFrame(Z)
del Z
gc.collect()

In [ ]:
winner_peaks = []
for celltype in celltypes:
    winners = allPeaks.iloc[np.argwhere(np.argmax(allPeaks, axis = 1) == np.argwhere(allPeaks.columns.isin([celltype]))[0])[:,0],:]
    winners = winners[(winners[celltype] / winners.drop(columns=celltype).max(axis = 1)) >= 2]
    winner_peaks.extend(winners[celltype].nlargest(500).index.tolist())

In [ ]:
# Onehot for winners
X_winners = X[np.argwhere(peaks.index.isin(winner_peaks))[:,0],:,:]

In [ ]:
# Get peak winners
peak_winners = peaks[peaks.index.isin(winner_peaks)].copy()
peak_winners['index'] = peak_winners.index
peak_winners = peak_winners.reset_index()
peak_winners['torch_index'] = peak_winners.index

In [ ]:
# Create results dictionary
folds = [0, 1, 2, 3, 4]
result_dict = {}
for celltype in celltypes:
    result_dict[celltype] = {}
    for midx in range(len(hits)):
        result_dict[celltype][midx] = {}
        for fold in folds:
            result_dict[celltype][midx][fold] = { 'before': np.array([]), 'after': np.array([])}

In [ ]:
n_resamples = 3
window_size = 500
with torch.no_grad():
    # Loop over motifs
    for midx in tqdm(range(len(hits_filtered))):
        selectedMotif = hits[midx]
        selectedMotif = selectedMotif[selectedMotif['sequence_name'].isin(peak_winners['index'])]
        if len(selectedMotif) > 0:
            selectedMotif = pd.merge(selectedMotif, peak_winners[["index", "torch_index"]], left_on="sequence_name", right_on="index")
            X_original = X_winners[selectedMotif['torch_index'],:,:]
            
            # Perturb all original sequences
            X_perturb = torch.empty(len(X_original), n_resamples, 4, 2114)
            selectedMotif['start'] = selectedMotif['start'] + (1057 - (window_size//2))
            selectedMotif['end'] = selectedMotif['end'] + (1057 - (window_size//2))
            annotation_tensor = torch.tensor(np.array(selectedMotif[['sequence_name', 'start', 'end']]))
            for i, (idx, start, end) in enumerate(annotation_tensor):
                X_perturb[i,:,:,:] = shuffle_sequences(X_original[i,:,:].unsqueeze(0), start=start, end=end, n=n_resamples, random_state=None)

            # Loop over cell types
            for celltype in celltypes:   
                # Predict for the perturbed sequences
                N, R, C, L = X_perturb.shape
                X_flat = X_perturb.permute(1, 0, 2, 3).reshape(R * N, C, L)
                preds = []
                for i in range(0, R * N, batch_size):
                    end = i+batch_size
                    if end > len(X_flat):
                        end = len(X_flat)
                    batch = X_flat[i:end,:,:].to('cuda', non_blocking=True)
                    _, pred, _ = custom_forward(model_dict[celltype], batch)
                    preds.append(pred)
                preds_flat = torch.cat(preds, dim=0)
                out_shape = preds_flat.shape[1:]
                preds_RN = preds_flat.reshape(R, N, *out_shape)
                result_dict[celltype][midx][fold]['after'] = preds_RN.squeeze()

                # Predict for original sequences and store
                before_fold = []
                for i in range(0, len(X_original), batch_size):
                    end = i+batch_size
                    if end > len(X_original):
                        end = len(X_original)
                    batch = X_original[i : end,:,:].to('cuda')
                    _, pred, _ = custom_forward(model_dict[celltype], batch.to(torch.float32))
                    before_fold.append(pred)
                result_dict[celltype][midx][fold]['before'] = torch.cat(before_fold, dim=0).cpu().numpy().squeeze()

    # Output after each fold
    with open(f"{chrombpnet_dir}/data/motif_activity_highvar.pkl", "wb") as f:
        pickle.dump(result_dict, f)

In [ ]:
for midx in range(len(hits_filtered)):
    ma = []
    for celltype in celltypes:
        if len(result_dict[celltype][midx][4]['after']) > 0:
            ma.append((result_dict[celltype][midx][4]['before'] - result_dict[celltype][midx][4]['after'].cpu().numpy()).mean())
        else:
            ma.append(np.nan)

    # Structure the data
    tmp_df = pd.DataFrame(np.array(ma), index = celltypes)
    tmp_df = tmp_df.rename(columns = {0: hits_filtered[midx]['motif_name'][0].split(" ")[0]}).T

    # Concat
    if midx == 0:
        ma_df = tmp_df
    else:
        ma_df = pd.concat([ma_df, tmp_df])

In [ ]:
ma_filtered = ma_df.copy()
ma_filtered = ma_filtered.dropna()
ma_filtered = ma_filtered[~ma_filtered.index.str.contains("ZNF")]
ma_filtered = ma_filtered[~ma_filtered.index.str.contains("C2H2")]
ma_filtered

In [ ]:
without_cycling = list(set(celltypes) - set(['cycling']))
Z = ((ma_filtered.T - ma_filtered.T.mean()) / ma_filtered.T.std()).T
Z = Z.drop(columns=["cycling"])
winners = np.argmax(Z, axis = 1)
allwinners = []
for celltype in without_cycling:
    winner_motifs = Z.iloc[np.argwhere(winners == np.argwhere(Z.columns.isin([celltype]))[0])[:,0],:].copy()
    winner_motifs = winner_motifs[winner_motifs[celltype] >= 1.0]
    allwinners.extend((winner_motifs[celltype] - winner_motifs.drop(columns=celltype).max(axis = 1)).nlargest(3).index.tolist())

In [ ]:
for midx in range(len(hits)):
    motif_name = list(motifs.keys())[midx].split(" ")[0].split(":")[0]
    if motif_name == "AC0558":
        break
PBX1_hits = hits[midx]

In [ ]:
protein = pd.read_csv(f"{chrombpnet_dir}/PBX1.protein.csv")
protein = protein.fillna(value = "ND")

In [ ]:
sns.boxplot(x = "meta", y = "feature", data = protein)
plt.savefig(f"{chrombpnet_dir}/PBX1_protein.pdf", dpi=300, bbox_inches="tight")
plt.close()

In [ ]:
import pyranges as pr

In [ ]:
#### GWAS
GWAS = pd.read_csv(f"{chrombpnet_dir}/GWAs/Suzuki.ALL-T2D.sumstats", sep = "\t")
GWAS = GWAS.replace([np.inf, -np.inf], np.nan)
GWAS = GWAS.dropna()

In [ ]:
GWAS_gr = pr.from_dict({'Chromosome': "chr" + GWAS['CHR'].astype(str), 'Start': GWAS['BP'], 'End': GWAS['BP'] + 1, 'Pvalue': GWAS['PVALASSOCIATION']})

In [ ]:
peaks = pd.read_csv(f"{chrombpnet_dir}/data/peaks/correlated/hits/beta_peaks.csv", index_col=0)
foreground_peaks = peaks[peaks["class"].str.contains(r"PC9_|PC4_|PC10_|PC11_|PC13_")]
background_peaks = peaks[~peaks["class"].str.contains(r"PC9_|PC4_|PC10_|PC11_|PC13_")]
baseline_peaks = background_peaks[background_peaks["class"].str.contains(r"baseline")]
background_peaks = background_peaks[~background_peaks["class"].str.contains(r"baseline")]

In [ ]:
foreground_bed = pd.DataFrame({'chromosome': foreground_peaks['chromosome'], 'start': foreground_peaks['start'], 'end': foreground_peaks['end']})
X_sig = extract_loci(foreground_bed, genome_file, ignore=['N','B', 'D', 'H', 'V', 'R','Y', 'S', 'W', 'K', 'M'], in_window = 500)
foreground_bed = foreground_bed.loc[np.array(foreground_bed.index.tolist())[np.argwhere(np.array(X_sig.sum(axis = 1).sum(axis = 1) == 500))[:,0]].tolist()]
foreground_bed = foreground_bed.reset_index(drop = True)
X_sig = extract_loci(foreground_bed, genome_file, ignore=['N','B', 'D', 'H', 'V', 'R','Y', 'S', 'W', 'K', 'M'], in_window = 500)
hits = fimo(motif_path, X_sig, threshold=0.001)
hits = hits[midx]
hits = hits.set_index(pd.Index(hits['sequence_name']))
hits = pd.merge(hits, foreground_bed, right_index=True, left_index=True, how="left")
foreground_hits_gr = pr.from_dict({'Chromosome': hits['chromosome'], 'Start': hits['start_y'] + hits['start_x'], 'End': hits['start_y'] + hits['end_x'], 'Score': hits['score']})
foreground_joined = GWAS_gr.join(foreground_hits_gr)
foreground_joined = foreground_joined.df

In [ ]:
background_bed = pd.DataFrame({'chromosome': background_peaks['chromosome'], 'start': background_peaks['start'], 'end': background_peaks['end']})
X_sig = extract_loci(background_bed, genome_file, ignore=['N','B', 'D', 'H', 'V', 'R','Y', 'S', 'W', 'K', 'M'], in_window = 500)
background_bed = background_bed.loc[np.array(background_bed.index.tolist())[np.argwhere(np.array(X_sig.sum(axis = 1).sum(axis = 1) == 500))[:,0]].tolist()]
background_bed = background_bed.reset_index(drop = True)
X_sig = extract_loci(background_bed, genome_file, ignore=['N','B', 'D', 'H', 'V', 'R','Y', 'S', 'W', 'K', 'M'], in_window = 500)
hits = fimo(motif_path, X_sig, threshold=0.001)
hits = hits[midx]
hits = hits.set_index(pd.Index(hits['sequence_name']))
hits = pd.merge(hits, background_bed, right_index=True, left_index=True, how="left")
background_hits_gr = pr.from_dict({'Chromosome': hits['chromosome'], 'Start': hits['start_y'] + hits['start_x'], 'End': hits['start_y'] + hits['end_x'], 'Score': hits['score']})
background_joined = GWAS_gr.join(background_hits_gr)
background_joined = background_joined.df

In [ ]:
baseline_bed = pd.DataFrame({'chromosome': baseline_peaks['chromosome'], 'start': baseline_peaks['start'], 'end': baseline_peaks['end']})
X_sig = extract_loci(baseline_bed, genome_file, ignore=['N','B', 'D', 'H', 'V', 'R','Y', 'S', 'W', 'K', 'M'], in_window = 500)
baseline_bed = baseline_bed.loc[np.array(baseline_bed.index.tolist())[np.argwhere(np.array(X_sig.sum(axis = 1).sum(axis = 1) == 500))[:,0]].tolist()]
baseline_bed = baseline_bed.reset_index(drop = True)
X_sig = extract_loci(baseline_bed, genome_file, ignore=['N','B', 'D', 'H', 'V', 'R','Y', 'S', 'W', 'K', 'M'], in_window = 500)
hits = fimo(motif_path, X_sig, threshold=0.001)
hits = hits[midx]
hits = hits.set_index(pd.Index(hits['sequence_name']))
hits = pd.merge(hits, baseline_bed, right_index=True, left_index=True, how="left")
baseline_hits_gr = pr.from_dict({'Chromosome': hits['chromosome'], 'Start': hits['start_y'] + hits['start_x'], 'End': hits['start_y'] + hits['end_x'], 'Score': hits['score']})
baseline_joined = GWAS_gr.join(baseline_hits_gr)
baseline_joined = baseline_joined.df

In [ ]:
counter = 0

In [ ]:
foreground_joined_subset = foreground_joined[foreground_joined['Score'] >= 7]
background_joined_subset = background_joined[background_joined['Score'] >= 7]
baseline_joined_subset = baseline_joined[baseline_joined['Score'] >= 7]
arr = GWAS['PVALASSOCIATION'].to_numpy()
nrandom = 100
dfs = []

for threshold in [1e-4, 1e-5, 1e-6, 1e-7, 1e-8, 5e-8]:
    # Foreground
    exp = []
    for i in range(nrandom):
        exp.append(((np.random.choice(arr, size=len(foreground_joined_subset), replace=False) <= threshold).sum() + 1) / (len(foreground_joined_subset) + 1))
    foreground_obs = (((foreground_joined_subset['Pvalue'] <= threshold).sum() +1) / (len(foreground_joined_subset) + 1))
    foreground_exp_mean = np.array(exp).mean()
    foreground_exp_std = np.std(np.array(exp))
    dfs.append(pd.DataFrame({'Type': 'Foreground', 'Threshold': threshold, 'Obs': foreground_obs, 'Exp': foreground_exp_mean, 'Std': foreground_exp_std}, index = [counter]))
    counter += 1
    
    # Background
    exp = []
    for i in range(nrandom):
        exp.append(((np.random.choice(arr, size=len(background_joined_subset), replace=False) <= threshold).sum() + 1) / (len(foreground_joined_subset) + 1))
    background_obs = (((background_joined_subset['Pvalue'] <= threshold).sum() +1) / (len(background_joined_subset) + 1))
    background_exp_mean = np.array(exp).mean()
    background_exp_std = np.std(np.array(exp))
    dfs.append(pd.DataFrame({'Type': 'Background', 'Threshold': threshold, 'Obs': background_obs, 'Exp': background_exp_mean, 'Std': background_exp_std}, index = [counter]))
    counter += 1

    # Baseline
    exp = []
    for i in range(nrandom):
        exp.append(((np.random.choice(arr, size=len(baseline_joined_subset), replace=False) <= threshold).sum() + 1)/ (len(foreground_joined_subset) + 1))
    baseline_obs = (((baseline_joined_subset['Pvalue'] <= threshold).sum() +1) / (len(baseline_joined_subset) + 1))
    baseline_exp_mean = np.array(exp).mean()
    baseline_exp_std = np.std(np.array(exp))
    dfs.append(pd.DataFrame({'Type': 'Baseline', 'Threshold': threshold, 'Obs': baseline_obs, 'Exp': baseline_exp_mean, 'Std': baseline_exp_std}, index = [counter]))
    counter += 1

In [ ]:
GWAS_enrichment = pd.concat(dfs, ignore_index=True)
GWAS_enrichment['OE'] = np.log2(GWAS_enrichment['Obs'] / GWAS_enrichment['Exp'])
GWAS_enrichment = GWAS_enrichment[GWAS_enrichment['Type'] != "Background"]

In [ ]:
sns.barplot(x = 'Threshold', y = 'OE', hue = 'Type', data = GWAS_enrichment)
plt.savefig(f"{chrombpnet_dir}/PBX1_motif_GWAS.pdf", dpi=300, bbox_inches="tight")
plt.close()